# **Start Section:**

In [1]:
# FIX K1: same notebook runs on Google Colab (Drive) or Kaggle (internet on).
# On Kaggle the working dir is /kaggle/working, so the original "./drive/MyDrive/..." paths
# below are reused unchanged; the dataset is copied there as train.csv / test.csv.
import os, glob, shutil
IN_KAGGLE = os.path.exists("/kaggle/working")
if IN_KAGGLE:
    _data_dir = "./drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/data"
    os.makedirs(_data_dir, exist_ok=True)
    shutil.copy(glob.glob("/kaggle/input/**/d04_a.csv", recursive=True)[0], f"{_data_dir}/train.csv")  # water_quality_wqi train
    shutil.copy(glob.glob("/kaggle/input/**/d04_b.csv", recursive=True)[0], f"{_data_dir}/test.csv")   # water_quality_wqi test
    print("Kaggle: water_quality_wqi data copied to", _data_dir)
else:
    from google.colab import drive
    drive.mount('/content/drive')


Kaggle: water_quality_wqi data copied to ./drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/data


In [2]:
# FIX K1: original Colab install block kept as-is (runs only on Colab).
# On Kaggle: install only the packages the Kaggle image lacks, pinned to the Colab versions,
# and do NOT restart the kernel with os._exit(0).
if IN_KAGGLE:
    !pip install -q optuna==4.6.0 optunahub==0.4.0 cmaes==0.12.0 xgboost==3.1.2 catboost==1.2.8 gpboost==1.6.1 ngboost==0.5.8 pgbm==2.2.0 pytorch-tabnet2==4.5.3 interpret==0.7.4 lime==0.2.0.1 mapie==0.6.0 puncc==0.8.0 properscoring==0.1 XlsxWriter==3.2.9
else:
    import os
    os.environ["PIP_CONSTRAINT"] = "/tmp/numpy_constraint.txt"
    !echo "numpy==1.26.4" > /tmp/numpy_constraint.txt

    !pip uninstall -y torch torchaudio torchvision \
        torchao torchcodec torchdata torchtune torchsummary -q 2>/dev/null

    !pip uninstall -y tensorflow tensorflow-text tensorflow-hub tf-keras \
        tensorflow_decision_forests tensorflow-probability \
        tensorflow-Datasets tensorflow-metadata -q 2>/dev/null

    !pip uninstall -y numpy scikit-learn shap xgboost lightgbm dask \
        seaborn plotly openpyxl Cython catboost interpret lime -q 2>/dev/null

    !pip install numpy==1.26.4 -q
    !pip install scikit-learn==1.6.1 -q
    !pip install torch==2.9.0 -q

    !pip install lightgbm==4.6.0 -q
    !pip install xgboost==3.1.2 -q
    !pip install catboost==1.2.8 -q
    !pip install gpboost==1.6.1 -q
    !pip install ngboost==0.5.8 -q
    !pip install pgbm==2.2.0 -q
    !pip install pytorch-tabnet2==4.5.3 -q

    !pip install bayesian-optimization==3.2.0 -q
    !pip install optuna==4.6.0 -q
    !pip install optunahub==0.4.0 -q
    !pip install cmaes==0.12.0 -q

    !pip install shap==0.44.0 -q
    !pip install lime==0.2.0.1 -q
    !pip install interpret==0.7.4 -q

    !pip install mapie==0.6.0 -q
    !pip install puncc==0.8.0 -q
    !pip install skorch==1.3.1 -q
    !pip install properscoring==0.1 -q

    !pip install dask[dataframe]==2025.12.0 -q
    !pip install cython==3.0.12 -q
    !pip install seaborn==0.13.2 -q
    !pip install plotly==5.24.1 -q
    !pip install kaleido==1.2.0 -q
    !pip install openpyxl==3.1.5 -q
    !pip install XlsxWriter==3.2.9 -q
    !pip install cp==2020.12.3 -q

    !pip install numpy==1.26.4 --force-reinstall --no-deps -q

    os._exit(0)


  Installing build dependencies ... done


  Getting requirements to build wheel ... done


  Preparing metadata (pyproject.toml) ... done


  Installing build dependencies ... done


  Getting requirements to build wheel ... done


  Preparing metadata (pyproject.toml) ... done


  Installing build dependencies ... done


  Getting requirements to build wheel ... done


  Preparing metadata (pyproject.toml) ... done


# **Imports**

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ngboost
from scipy.stats import randint
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.svm import SVR
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from gpboost import GPBoostRegressor
from ngboost import NGBRegressor
import optuna
import optunahub
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingGridSearchCV
from interpret import show
from interpret.blackbox import LimeTabular, ShapKernel
from optuna.samplers import RandomSampler
import random
import time
from ngboost.distns import Normal
from ngboost.scores import LogScore
from scipy.stats import norm
from optuna.samplers import BaseSampler
from optuna.samplers import GridSampler
from optuna.samplers import TPESampler
from optuna.samplers import PartialFixedSampler
from optuna.samplers import CmaEsSampler
from optuna.samplers import QMCSampler
from optuna.samplers import NSGAIIISampler
from optuna.samplers import NSGAIISampler
from optuna.samplers import BruteForceSampler
from optuna.samplers import GPSampler
from interpret import set_visualize_provider
from interpret.provider import InlineProvider
from interpret.glassbox import ExplainableBoostingRegressor
from interpret import show
import plotly.express as px
from io import BytesIO
from openpyxl import Workbook, load_workbook
import os
import properscoring as ps
import io
from openpyxl.drawing.image import Image as openpyxlImage
import warnings
import xlsxwriter
from openpyxl.drawing.image import Image
from pgbm.sklearn import HistGradientBoostingRegressor
import torch
from pgbm.torch import PGBM
import plotly.graph_objects as go
warnings.filterwarnings('ignore')
import pickle
import json
from mapie.subsample import Subsample
from mapie.regression import MapieRegressor
from deel.puncc.metrics import regression_sharpness, regression_mean_coverage
from deel.puncc.api.prediction import BasePredictor, DualPredictor
from deel.puncc.regression import SplitCP, CVPlus, CQR
from deel.puncc.plotting import plot_prediction_intervals
from sklearn.model_selection import train_test_split
from typing_extensions import TypedDict
from typing import Union
from mapie.metrics import regression_coverage_score
from sklearn.model_selection import KFold
from PIL import Image as PImage
from pytorch_tabnet import TabNetRegressor

[1/2] c++ -MMD -MF splitgain_cpu.o.d -DTORCH_EXTENSION_NAME=split_decision -DTORCH_API_INCLUDE_EXTENSION_H -isystem /usr/local/lib/python3.13/dist-packages/torch/include -isystem /usr/local/lib/python3.13/dist-packages/torch/include/torch/csrc/api/include -isystem /usr/include/python3.13 -fPIC -std=c++17 -c /usr/local/lib/python3.13/dist-packages/pgbm/torch/splitgain_cpu.cpp -o splitgain_cpu.o 


[2/2] c++ splitgain_cpu.o -shared -L/usr/local/lib/python3.13/dist-packages/torch/lib -lc10 -ltorch_cpu -ltorch -ltorch_python -o split_decision.so


I0000 00:00:1795404104.528027      33 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


I0000 00:00:1795404123.924785      33 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [4]:
# Go to find & replace button and replace (water_quality_wqi_uncertainty_analysis) with your folder name. Rename your train and test Dataset as train.csv and test.csv.
# Modify the names of the feature in the below cell.
# Replace (str) with actual data label name.

In [5]:
feature_names = ['R1', 'R2', 'R3', 'R4', 'R5', 'R6', 'R7', 'R8', 'R9', 'R10', 'R11', 'R12', 'R13', 'R14', 'R15', 'R16', 'R17', 'R18', 'R19', 'R20', 'R21', 'R22', 'R23', 'R24', 'R25', 'R26', 'R27', 'R28', 'R29', 'R30', 'R31', 'R32', 'R33', 'R34', 'R35', 'R36', 'R37', 'R38', 'R39', 'R40', 'R41', 'R42', 'R43', 'R44', 'R45', 'R46', 'R47', 'R48', 'R49', 'R50', 'R51', 'R52', 'R53', 'R54', 'R55', 'R56', 'R57', 'R58', 'R59', 'R60', 'R61', 'R62', 'R63', 'R64', 'R65', 'R66', 'R67', 'R68', 'R69', 'R70', 'R71', 'R72', 'R73', 'R74', 'R75', 'R76', 'R77', 'R78', 'R79', 'R80', 'R81', 'R82', 'R83', 'R84', 'R85', 'R86', 'R87', 'R88', 'R89', 'R90', 'R91', 'R92', 'R93', 'R94', 'R95', 'R96', 'R97', 'R98', 'R99', 'R100', 'R101', 'R102', 'R103', 'R104', 'R105', 'R106', 'R107', 'R108', 'R109', 'R110', 'R111', 'R112', 'R113', 'R114', 'R115', 'R116', 'R117', 'R118', 'R119', 'R120', 'R121', 'R122', 'R123', 'R124', 'R125', 'R126', 'R127', 'R128', 'R129', 'R130', 'R131', 'R132', 'R133', 'R134', 'R135', 'R136', 'R137', 'R138', 'R139', 'R140', 'R141', 'R142', 'R143', 'R144', 'R145', 'R146', 'R147', 'R148', 'R149', 'R150', 'R151']
TARGET_LABEL = 'WQI'  # FIX B12: target name used in plot labels (was 'str', the concrete dataset's target, on every dataset)

In [6]:
train_data_path = "./drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/data/train.csv"
test_data_path = "./drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/data/test.csv"
train_data = pd.read_csv(train_data_path)
test_data = pd.read_csv(test_data_path)
print("Training data loaded successfully.")
print("Test data loaded successfully.")

Training data loaded successfully.
Test data loaded successfully.


In [7]:
print("\nShape of training data:", train_data.shape)
print("First 5 rows of training data:\n", train_data.head(5))
print("\nShape of test data:", test_data.shape)
print("First 5 rows of test data:\n", test_data.head(5))


Shape of training data: (190, 152)
First 5 rows of training data:
        R1      R2      R3      R4      R5      R6      R7      R8      R9  \
0  3.5960  3.6080  3.0100  3.6130  3.3760  2.2780  1.3410  3.4570  3.3540   
1  2.9540  3.0470  3.1160  4.2640  4.4090  3.4810  2.3760  3.4030  3.4320   
2  2.4870  2.9430  2.5290  3.7280  2.9870  1.8890  1.9960  2.1020  1.6180   
3  1.3640  1.4186  1.3789  1.4776  1.5063  1.4047  1.2648  1.5166  1.5443   
4  1.3074  1.3820  1.3476  1.3986  1.3403  1.3004  1.3719  1.3984  1.3865   

      R10  ...    R143    R144    R145    R146    R147   R148    R149    R150  \
0  1.9900  ...  2.7930  1.9660  2.3070  1.5280  1.9380  3.371  2.2060  1.5220   
1  2.6350  ...  3.0260  2.3530  2.0400  1.7400  2.6250  4.026  2.9010  1.7450   
2  1.2720  ...  1.2760  1.5990  1.8610  1.0290  1.8690  3.905  2.3200  1.3310   
3  1.3834  ...  1.2223  1.0919  1.0995  1.0856  1.1728  1.315  1.2341  1.0753   
4  1.3964  ...  1.1691  1.0689  1.1230  1.1384  1.1810  1.235  1

In [8]:
X_train = train_data.iloc[:, :-1]
y_train = train_data.iloc[:, -1]
X_test = test_data.iloc[:, :-1]
y_test = test_data.iloc[:, -1]
# FIX D4 / D5 (approved 2026-11-22): settings that follow the test-set size
N_BINS = max(2, min(10, len(y_test) // 10))  # D4: bins for binned metrics (was 10 on every dataset; pile: 2-3 points per bin)
COP_T_BURNIN = min(50, len(y_test) // 2)  # D5: COP burn-in (was 50, so with < 50 test points COP's correction never started)
x_test= X_test
print("\nShape of X_train:", X_train.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of X_test:", X_test.shape)
print("Shape of y_test:", y_test.shape)


Shape of X_train: (190, 151)
Shape of y_train: (190,)
Shape of X_test: (82, 151)
Shape of y_test: (82,)


In [9]:
# Apply z-score normalization
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Print the first five rows of the normalized data
print("\nFirst five rows of normalized X_train:")
print(X_train[:5])

print("\nFirst five rows of normalized X_test:")
print(X_test[:5])


First five rows of normalized X_train:
[[ 1.22103295  1.13089815  0.61081314  0.73682842  0.54673262 -0.0261968
  -0.51531151  0.70303503  0.5510272  -0.2441054   0.17648144  0.69219002
   0.9485328   0.82481901  0.34566436 -0.40609522 -0.463771    0.43872889
   0.89421381  1.05057715  1.04529849  1.03900179  0.42575888  0.173456
   0.45642529  0.83037026  0.89010153  0.77972274  0.71866378  0.6206252
   0.82167079  0.97412629  0.67298061  0.39758814  0.52679374  0.94375937
   0.74197071  0.86577935  1.27064247  1.51437415  0.46596778 -0.30888056
  -0.12461695  0.49591603  0.72957494  0.5811918   0.31848773  0.53466849
   0.20246884  0.11900459  0.99635035  0.72105768  0.43065398  0.36777539
   0.10638109 -0.0263905   0.2318847   0.82337505  0.74363607  0.85248236
   0.71669043 -0.11909161 -0.10843386  0.51688952  0.84551287 -0.1576873
  -0.12716017  0.37884167  0.35396679 -0.24556803  0.07962918  0.82628894
   0.54494889  1.02963853  1.17379609  0.60350175 -0.17692428 -0.31350223
   

# **Functions:**

In [10]:
# Define the model classes
model_classes = {
    'Random Forest': RandomForestRegressor,
    'Gradient Boosting': GradientBoostingRegressor,
    'XGBoost': XGBRegressor,
    'LightGBM': LGBMRegressor,
    'GPBoost': GPBoostRegressor,
    'CatBoost': CatBoostRegressor,
    'HistGradientBoosting': HistGradientBoostingRegressor,
    'TabNet': TabNetRegressor,
    'NGBoost': NGBRegressor
}

def _ensure_excel_file(path):
    import os
    import pandas as pd
    if not os.path.exists(path):
        parent = os.path.dirname(path)
        if parent:
            os.makedirs(parent, exist_ok=True)
        pd.DataFrame().to_excel(path)
    return path

In [11]:
def get_best_model_params(results, model_name):
    # Map model names to dictionary keys, assuming keys are strings like 'XGBoost' and not objects
    model_keys = {
        'LightGBM': 'LightGBM',
        'XGBoost': 'XGBoost',
        'GPBoost': 'GPBoost',
        'NGBoost': 'NGBoost',
        'GradientBoosting': 'Gradient Boosting',
        'HGBR' : 'HistGradientBoosting',
        'CatBoost' : 'CatBoost',
        'TabNet' : 'TabNet',
        'PGBM' : 'PGBM'
    }

    # Ensure the requested model name is valid
    if model_name not in model_keys:
        raise ValueError(f"Model name '{model_name}' is not recognized. Available models are: {list(model_keys.keys())}")

    # Filter out entries for the specified model
    model_entries = {key: value for key, value in results.items() if key[0] == model_keys[model_name]}

    # Find the entry with the best (lowest) 'best_score'
    best_entry_key, best_entry_value = min(model_entries.items(), key=lambda item: item[1]['best_score'])

    # Return the best hyperparameters
    return best_entry_value['best_params']

In [12]:
# FIX Q2: quantile crossing. Each quantile level is a separate model, so for some rows the
# predictions can come out in the wrong order (e.g. q0.10 < q0.05). Before any metric we sort
# each row's quantile predictions (monotone rearrangement, Chernozhukov et al. 2010) and record
# how many rows crossed, for the Excel report.
quantile_crossing_report = {}

def rearrange_quantiles(predictions_df, tag=None):
    q_cols = sorted(c for c in predictions_df.columns if c != 'Actual')
    values = predictions_df[q_cols].to_numpy(dtype=float)
    crossed = (np.diff(values, axis=1) < 0).any(axis=1)
    if tag is not None:
        quantile_crossing_report[tag] = {'rows': int(len(values)), 'rows_crossing': int(crossed.sum()),
                                         'pct_crossing': round(100.0 * crossed.mean(), 2)}
    out = predictions_df.copy()
    out[q_cols] = np.sort(values, axis=1)
    return out


In [13]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def quantile_regression_print(predictions_df, file_path):
    predictions_df = rearrange_quantiles(predictions_df, file_path)  # FIX Q2
    # Extract actual values
    actual_values = predictions_df['Actual'].values

    # Debugging: Print the first few actual values
    print("First few actual values:", actual_values[:5])

    # Calculate coverage for the specific interval (0.05 to 0.95)
    lower_quantile = 0.05
    median_quantile = 0.5
    upper_quantile = 0.95

    lower_preds = predictions_df[lower_quantile].values
    median_preds = predictions_df[median_quantile].values
    upper_preds = predictions_df[upper_quantile].values

    in_interval = ((actual_values >= lower_preds) & (actual_values <= upper_preds))
    coverage = in_interval.mean()
    print(f"Coverage of 90% prediction interval: {coverage * 100:.2f}%")

    # Create a new Excel workbook and add a worksheet
    wb = Workbook()
    ws = wb.active
    ws.title = "quantile_regression_print"

    # Plot actual vs. all predicted quantiles
    num_points = len(predictions_df)
    indices = np.arange(num_points)

    plt.figure(figsize=(12, 6))
    plt.plot(indices, actual_values[:num_points], label=f'Actual {TARGET_LABEL}', marker='o', linestyle='-', color='black')

    # Loop over all columns assumed to be quantile predictions
    for column in predictions_df.columns:
        if column != 'Actual':  # Skip the actual values column
            plt.plot(indices, predictions_df[column][:num_points],
                     label=f'Predicted Quantile {column}', linestyle='--')

    plt.xlabel('Sample Number')
    plt.ylabel(f'{TARGET_LABEL}')
    plt.title(f'Actual vs. Predicted Quantile {TARGET_LABEL}')
    plt.legend(loc='upper left', fontsize='small', bbox_to_anchor=(1.05, 1), borderaxespad=0.)

    # Save the plot to a BytesIO object
    img_data = BytesIO()
    img_data
    plt.savefig(img_data, format='png', bbox_inches='tight')
    plt.close()
    img_data.seek(0)

    # Insert the image into the Excel sheet
    img = Image(img_data)
    ws.add_image(img, 'A1')

    # Plot actual values and specific prediction intervals (0.05, 0.5, 0.95)
    plt.figure(figsize=(12, 6))
    plt.plot(indices, actual_values, label=f'Actual {TARGET_LABEL}', marker='o', linestyle='-', color='black')  # Connect points with a line
    plt.plot(indices, median_preds, label=f'Median Prediction ({median_quantile})', marker='x', linestyle='-', color='blue')
    plt.fill_between(
        indices,
        lower_preds,
        upper_preds,
        color='blue',
        alpha=0.5,
        label=f'Prediction Interval ({lower_quantile}-{upper_quantile})'
    )
    plt.xlabel('Sample Number')
    plt.ylabel(f' {TARGET_LABEL}')
    plt.title(f'Actual {TARGET_LABEL} with Prediction Intervals')
    plt.legend(loc='upper left', fontsize='small', bbox_to_anchor=(1, 1))

    # Save the second plot to another BytesIO object
    img_data = BytesIO()
    img_data
    plt.savefig(img_data, format='png', bbox_inches='tight')
    plt.close()
    img_data.seek(0)

    # Insert the second image into the Excel sheet
    img = Image(img_data)
    ws.add_image(img, 'A30')  # Adjust the cell position as needed

    # Save the workbook
    _ensure_parent_dir(file_path)
    wb.save(_ensure_parent_dir(file_path))

In [14]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def evaluation_features_print(predictions_df, X_test, file_path):
    predictions_df = rearrange_quantiles(predictions_df, file_path)  # FIX Q2
    # Ensure X_test is a DataFrame
    if isinstance(X_test, np.ndarray):
        X_test = pd.DataFrame(X_test, columns=feature_names)
    X_test.columns = feature_names

    # Get quantile columns from predictions_df except 'Actual'
    quantile_columns = [col for col in predictions_df.columns if col != 'Actual']

    # Check if the Excel file already exists
    if os.path.exists(file_path):
        wb = load_workbook(_ensure_excel_file(file_path))
        ws = wb.create_sheet(title="New Evaluation Features")
    else:
        wb = Workbook()
        ws = wb.active
        ws.title = "evaluation_features_print"

    image_row = 1  # Where to start images

    for feature_to_plot in feature_names:
        feature_values = X_test[feature_to_plot]
        plot_df = predictions_df.copy()
        plot_df[feature_to_plot] = feature_values.values

        # Sort by current feature to plot lines well
        plot_df = plot_df.sort_values(by=feature_to_plot)

        # Begin Matplotlib plot
        plt.figure(figsize=(6, 4))
        # Plot each quantile as a line
        for quantile in quantile_columns:
            plt.plot(plot_df[feature_to_plot], plot_df[quantile], label=f"{quantile}")

        # Plot actual values as scatter
        plt.scatter(plot_df[feature_to_plot], plot_df['Actual'], color='black', s=12, label=f"Actual {TARGET_LABEL}")

        plt.title(f"Predicted Quantiles vs. {feature_to_plot}")
        plt.xlabel(feature_to_plot)
        plt.ylabel(f"{TARGET_LABEL}")
        plt.legend(loc='center left', bbox_to_anchor=(1.0, 0.5))
        plt.tight_layout()

        # Save figure to a BytesIO stream
        img_bytes = BytesIO()
        img_bytes
        plt.savefig(img_bytes, format="png")
        plt.close()
        img_bytes.seek(0)

        img = Image(img_bytes)
        img.anchor = f"A{image_row}"
        ws.add_image(img)

        image_row += 20  # Adjust as you like

    _ensure_parent_dir(file_path)
    wb.save(_ensure_parent_dir(file_path))

In [15]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def evaluate_uncertainity_matrix(predictions_df, quantiles, excel_file_path, model_name='Model'):
    predictions_df = rearrange_quantiles(predictions_df, excel_file_path)  # FIX Q2

   # Extract the actual target values
    y_true = predictions_df['Actual']

    # Open existing Excel file
    wb = load_workbook(_ensure_excel_file(excel_file_path))

    # Create unique sheet names
    def get_unique_sheet_name(wb, base_name):
        sheet_name = base_name
        i = 1
        while sheet_name in wb.sheetnames:
            sheet_name = f"{base_name}_{i}"
            i += 1
        return sheet_name

    # Create separate sheets for plots and values
    plots_sheet_title = get_unique_sheet_name(wb, 'Plots')
    values_sheet_title = get_unique_sheet_name(wb, 'Values')

    ws_plots = wb.create_sheet(title=plots_sheet_title)
    ws_values = wb.create_sheet(title=values_sheet_title)
    # FIX Q2: how often the raw quantile predictions crossed (before sorting)
    _cr = quantile_crossing_report.get(excel_file_path, {})
    ws_values.append(['Rows with quantile crossing (before sorting)', _cr.get('rows_crossing'), 'of', _cr.get('rows'), 'rows', f"{_cr.get('pct_crossing')}%"])
    print(f"Quantile crossing before sorting: {_cr.get('rows_crossing')} of {_cr.get('rows')} rows ({_cr.get('pct_crossing')}%)")

    # Positions to place images
    image_positions = ['A1', 'A25', 'A49', 'A73']  # Adjust as needed

    ### 1. Validity / Coverage ###

    # Compute empirical coverage for each quantile
    empirical_quantiles = []
    for q in quantiles:
        coverage = (predictions_df[q] >= y_true).mean()
        empirical_quantiles.append(coverage)

    # Plot theoretical quantiles vs empirical quantiles
    plt.figure(figsize=(10, 6))
    sns.lineplot(x=quantiles, y=quantiles, color="magenta", linestyle='--', linewidth=2, label="Ideal")
    sns.lineplot(x=quantiles, y=empirical_quantiles, color="blue", linewidth=2, label="Empirical")
    sns.scatterplot(x=quantiles, y=empirical_quantiles, color="blue", s=50)
    plt.xlabel("Nominal Quantile Levels")
    plt.ylabel("Empirical Quantile Levels")
    plt.title(f"Validity / Coverage - {model_name}")
    plt.legend()

    # Save plot to buffer
    buffer = io.BytesIO()
    buffer
    plt.savefig(buffer, format='png')
    plt.close()
    buffer.seek(0)

    # Create an Image object
    img = openpyxlImage(buffer)
    img.width = 640
    img.height = 480

    # Add the image to the plots worksheet
    ws_plots.add_image(img, image_positions[0])

    ### 2. Sharpness / Interval Length ###

    # Define coverage levels to evaluate
    coverage_levels = [0.6, 0.8, 0.9]
    average_interval_lengths = []

    # Compute average interval lengths for each coverage level
    for c in coverage_levels:
        lower_q = (1 - c) / 2
        upper_q = 1 - lower_q
        # Find the nearest quantiles in our quantiles list
        lower_quantile = min(quantiles, key=lambda x: abs(x - lower_q))
        upper_quantile = min(quantiles, key=lambda x: abs(x - upper_q))

        # Compute interval length for each observation
        interval_length = predictions_df[upper_quantile] - predictions_df[lower_quantile]
        avg_interval_length = interval_length.mean()
        average_interval_lengths.append(avg_interval_length)
        # Write interval length information to the values worksheet
        ws_values.append([f'Coverage level: {c*100:.0f}%', f'Interval: [{lower_quantile}, {upper_quantile}]', f'Average Interval Length: {avg_interval_length:.4f}'])

    # Plot coverage levels vs average interval lengths
    plt.figure(figsize=(10, 6))
    plt.plot([c*100 for c in coverage_levels], average_interval_lengths, marker='o')
    plt.xlabel('Coverage Level (%)')
    plt.ylabel('Average Interval Length')
    plt.title(f'Sharpness / Interval Length - {model_name}')

    # Save plot to buffer
    buffer = io.BytesIO()
    buffer
    plt.savefig(buffer, format='png')
    plt.close()
    buffer.seek(0)

    # Create an Image object
    img = openpyxlImage(buffer)
    img.width = 640
    img.height = 480

    # Add image to the plots worksheet
    ws_plots.add_image(img, image_positions[1])

    ### 3. Negative Log-Likelihood (NLL) ###

    # Use the 5th and 95th percentiles to estimate standard deviation
    if 0.05 in quantiles and 0.95 in quantiles:
        z_lower = norm.ppf(0.05)  # z-score for 5% quantile (~ -1.6449)
        z_upper = norm.ppf(0.95)  # z-score for 95% quantile (~ +1.6449)

        # Extract quantile predictions
        q_lower = predictions_df[0.05]
        q_upper = predictions_df[0.95]

    elif 0.1 in quantiles and 0.9 in quantiles:
        z_lower = norm.ppf(0.1)  # z-score for 10% quantile (~ -1.2816)
        z_upper = norm.ppf(0.9)  # z-score for 90% quantile (~ +1.2816)

        # Extract quantile predictions
        q_lower = predictions_df[0.1]
        q_upper = predictions_df[0.9]
    else:
        raise ValueError("Required quantiles for NLL estimation not found in quantiles list.")

    # Use median as the mean estimate
    if 0.5 in quantiles:
        q_median = predictions_df[0.5]
    else:
        # If 0.5 quantile is not available, use the middle quantile
        q_median = predictions_df[quantiles[len(quantiles)/2]]

    # Estimate standard deviation for each observation
    std_estimates = (q_upper - q_lower) / (z_upper - z_lower)

    # Ensure standard deviations are positive and non-zero
    std_estimates = std_estimates.clip(lower=1e-6)

    # Extract mean estimates (median predictions)
    mean_estimates = q_median

    # Compute Negative Log-Likelihood for each observation
    nll = -norm.logpdf(y_true, loc=mean_estimates, scale=std_estimates)

    # Compute average NLL
    average_nll = nll.mean()
    # Write average NLL to the values worksheet
    ws_values.append(['Average Negative Log-Likelihood (NLL)', average_nll])

    # Write NLL values to the values worksheet
    ws_values.append(['NLL Values'])
    for value in nll:
        ws_values.append([value])

    # Plot ECDF of NLL values
    plt.figure(figsize=(10, 6))
    sns.ecdfplot(nll, color='blue', linewidth=2)
    plt.xlabel('Negative Log-Likelihood (NLL)')
    plt.ylabel('ECDF')
    plt.title(f'NLL ECDF - {model_name}')

    # Save plot to buffer
    buffer = io.BytesIO()
    buffer
    plt.savefig(buffer, format='png')
    plt.close()
    buffer.seek(0)

    # Create an Image object
    img = openpyxlImage(buffer)
    img.width = 640
    img.height = 480

    # Add image to the plots worksheet
    ws_plots.add_image(img, image_positions[2])

    ### 4. Continuous Ranked Probability Score (CRPS) ###

    # Prepare ensemble of quantile predictions for each observation
    ensemble_predictions = predictions_df[quantiles].to_numpy()

    # Ensure quantile predictions are sorted for each observation
    ensemble_predictions.sort(axis=1)

    # Compute CRPS for each observation using ensemble predictions
    # FIX Q4: CRPS from a set of quantiles = 2 * integral of the pinball loss over tau.
    # Each quantile level gets a weight equal to the share of [0, 1] it represents
    # (levels are unevenly spaced). Was: ps.crps_ensemble, which treats the 13 quantiles as
    # 13 equally likely samples.
    _taus = np.array(quantiles, dtype=float)
    _edges = np.concatenate([[0.0], (_taus[1:] + _taus[:-1]) / 2.0, [1.0]])
    _w = np.diff(_edges)
    _err = np.asarray(y_true, dtype=float)[:, None] - ensemble_predictions
    _pinball = np.maximum(_taus * _err, (_taus - 1.0) * _err)
    crps_values = 2.0 * (_pinball * _w).sum(axis=1)

    # Compute average CRPS
    average_crps = crps_values.mean()
    # Write average CRPS to the values worksheet
    ws_values.append(['Average Continuous Ranked Probability Score (CRPS)', average_crps])

    # Write CRPS values to the values worksheet
    ws_values.append(['CRPS Values'])
    for value in crps_values:
        ws_values.append([value])

    # Plot ECDF of CRPS values
    plt.figure(figsize=(10, 6))
    sns.ecdfplot(crps_values, color='green', linewidth=2)
    plt.xlabel('Continuous Ranked Probability Score (CRPS)')
    plt.ylabel('ECDF')
    plt.title(f'CRPS ECDF - {model_name}')

    # Save plot to buffer
    buffer = io.BytesIO()
    buffer
    plt.savefig(buffer, format='png')
    plt.close()
    buffer.seek(0)

    # Create an Image object
    img = openpyxlImage(buffer)
    img.width = 640
    img.height = 480

    # Add image to the plots worksheet
    ws_plots.add_image(img, image_positions[3])

    # Save the workbook
    _ensure_parent_dir(excel_file_path)
    wb.save(_ensure_parent_dir(excel_file_path))

    # Return a dictionary of results
    results = {
        'empirical_quantiles': empirical_quantiles,
        'average_interval_lengths': average_interval_lengths,
        'coverage_levels': coverage_levels,
        'average_nll': average_nll,
        'average_crps': average_crps,
        'nll_values': nll,
        'crps_values': crps_values
    }

    return results

In [16]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def calibrate_and_plot_intervals(predictions_df, excel_file_path, alpha=0.1, max_samples=1000, random_seed=42):
    predictions_df = rearrange_quantiles(predictions_df, excel_file_path)  # FIX Q2
    # Extract necessary columns
    lower = predictions_df[0.05].fillna(0)
    upper = predictions_df[0.95].fillna(0)
    pred = predictions_df[0.5].fillna(0)
    labels = predictions_df['Actual'].fillna(0)

    # Problem setup
    total_samples = labels.shape[0]
    n = min(max_samples, total_samples)  # Ensure n does not exceed the number of samples

    # Ensure there are enough samples for both calibration and validation
    if total_samples <= n:
        n = total_samples // 2

    # Split the data into calibration and validation sets
    np.random.seed(random_seed)  # Set the random seed for reproducibility
    idx = np.array([1] * n + [0] * (total_samples - n)) > 0
    np.random.shuffle(idx)
    cal_labels, val_labels = labels[idx], labels[~idx]
    cal_upper, val_upper = upper[idx], upper[~idx]
    cal_lower, val_lower = lower[idx], lower[~idx]
    cal_pred, val_pred = pred[idx], pred[~idx]

    # Calculate uncertainty intervals
    cal_U = cal_upper - cal_lower
    val_U = val_upper - val_lower

    # Avoid division by zero by replacing zero intervals with a small number
    cal_U[cal_U == 0] = np.finfo(float).eps
    val_U[val_U == 0] = np.finfo(float).eps

    # Get scores
    cal_scores = np.abs(cal_pred - cal_labels) / cal_U

    # Debugging: Check for NaN in scores
    if np.isnan(cal_scores).any():
        print("NaN values found in cal_scores. Check data integrity.")

    # Get the score quantile
    qhat = np.quantile(cal_scores, np.ceil((n + 1) * (1 - alpha)) / n, interpolation='higher')

    # Deploy (output=lower and upper adjusted quantiles)
    prediction_sets = [val_pred - val_U * qhat, val_pred + val_U * qhat]

    # Calculate empirical coverage (before and after calibration)
    prediction_sets_uncalibrated = [val_lower, val_upper]
    empirical_coverage_uncalibrated = ((val_labels >= prediction_sets_uncalibrated[0]) & (val_labels <= prediction_sets_uncalibrated[1])).mean() * 100
    print(f"The empirical coverage before calibration is: {empirical_coverage_uncalibrated:.2f}%")

    empirical_coverage = ((val_labels >= prediction_sets[0]) & (val_labels <= prediction_sets[1])).mean() * 100
    print(f"The empirical coverage after calibration is: {empirical_coverage:.2f}%")

    # Create a DataFrame with all the necessary data
    data = pd.DataFrame({
        'Index': np.arange(len(val_labels)),
        'Uncalibrated Lower': val_lower,
        'Uncalibrated Upper': val_upper,
        'Calibrated Lower': prediction_sets[0],
        'Calibrated Upper': prediction_sets[1],
        'True Label': val_labels,
        'Predicted Value': val_pred
    })

    plt.figure(figsize=(14, 8))

    # Plot the uncalibrated prediction intervals as a shaded area
    plt.fill_between(
        data['Index'],
        data['Uncalibrated Lower'],
        data['Uncalibrated Upper'],
        color='blue',
        alpha=0.5,
        label='Uncalibrated Prediction Interval'
    )

    # Plot the calibrated prediction intervals as a shaded area
    plt.fill_between(
        data['Index'],
        data['Calibrated Lower'],
        data['Calibrated Upper'],
        color='lightgreen',
        alpha=0.5,
        label='Calibrated Prediction Interval'
    )

    # Plot the true labels as points and connect them with a line
    plt.plot(
        data['Index'],
        data['True Label'],
        'o-',
        color='red',
        markersize=4,
        label='Actual Value'
    )

    # Plot the predicted values as points and connect them with a line
    plt.plot(
        data['Index'],
        data['Predicted Value'],
        'o-',
        color='black',
        markersize=4,
        label='Predicted Value'
    )

    plt.xlabel('Sample Number')
    plt.ylabel(f'{TARGET_LABEL}')
    plt.title('Calibrated and Uncalibrated Prediction Intervals with True and Predicted Values')
    plt.legend(loc='upper left', fontsize='small', bbox_to_anchor=(1.05, 1), borderaxespad=0.)
    plt.tight_layout()

    # Save the plot as an image
    image_path = 'plot.png'
    _ensure_parent_dir(image_path)
    plt.savefig(_ensure_parent_dir(image_path))
    plt.close()

    # Load the existing Excel file
    workbook = load_workbook(_ensure_excel_file(excel_file_path))

    # Create a new sheet for the plot
    new_sheet_name = "calibrate_and_plot_intervals"
    workbook.create_sheet(title=new_sheet_name)

    # Insert the image into the new sheet
    sheet = workbook[new_sheet_name]
    img = Image(image_path)
    sheet.add_image(img, 'A1')

    # Save the workbook
    _ensure_parent_dir(excel_file_path)
    workbook.save(_ensure_parent_dir(excel_file_path))

In [17]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def plot_quantile_intervals_to_excel(predictions_df, file_path, alpha=0.1):
    predictions_df = rearrange_quantiles(predictions_df, file_path)  # FIX Q2
    """
    Plots conformal prediction intervals and their coverage, and saves the plots to an Excel file.

    Parameters:
    - predictions_df: DataFrame containing prediction quantiles and actual values.
    - file_path: Path to the Excel file where plots will be saved.
    - alpha: Desired coverage level (default is 0.1).
    """
    # Extract mean prediction and uncertainty
    mean_prediction = predictions_df[0.5]
    uncertainty = predictions_df[0.95] - predictions_df[0.05]
    actual_values = predictions_df['Actual']

    # Calculate conformal scores
    conformal_scores = np.abs(mean_prediction - actual_values) / uncertainty

    # Function to find weighted quantile
    def weighted_quantile(values, quantile, sample_weight=None):
        values = np.array(values)
        if sample_weight is None:
            sample_weight = np.ones(len(values))
        sorter = np.argsort(values)
        values, sample_weight = values[sorter], sample_weight[sorter]
        weighted_quantiles = np.cumsum(sample_weight) - 0.5 * sample_weight
        weighted_quantiles /= np.sum(sample_weight)
        return np.interp(quantile, weighted_quantiles, values)

    # Calculate weighted quantile
    weights = np.ones_like(conformal_scores)  # Uniform weights for simplicity
    quantile = weighted_quantile(conformal_scores, 1 - alpha, sample_weight=weights)

    # Calculate prediction intervals
    lower_bound = mean_prediction - quantile * uncertainty
    upper_bound = mean_prediction + quantile * uncertainty

    # Naive conformal prediction
    naive_quantile = np.quantile(conformal_scores, 1 - alpha)
    naive_lower_bound = mean_prediction - naive_quantile * uncertainty
    naive_upper_bound = mean_prediction + naive_quantile * uncertainty

    # Coverage calculation (not plotted, just for potential diagnostics)
    weighted_coverage_points = (actual_values >= lower_bound) & (actual_values <= upper_bound)
    naive_coverage_points = (actual_values >= naive_lower_bound) & (actual_values <= naive_upper_bound)

    # Visualization using Matplotlib
    fig, ax = plt.subplots(figsize=(10, 5))

    x = np.arange(len(predictions_df))
    # Actual Values
    ax.plot(x, actual_values, label='Actual', color='black', linewidth=2)

    # Mean Prediction
    ax.plot(x, mean_prediction, label='Mean Prediction', color='darkorange', linewidth=2)

    # Weighted Prediction Interval (blue fill)
    ax.fill_between(x, lower_bound, upper_bound, color='blue', alpha=0.3, label='Weighted Interval')
    # Optionally, plot the bounds as lines if you want:
    # ax.plot(x, lower_bound, color='blue', linewidth=1, linestyle='--')
    # ax.plot(x, upper_bound, color='blue', linewidth=1, linestyle='--')

    # Naive Prediction Interval (red fill)
    ax.fill_between(x, naive_lower_bound, naive_upper_bound, color='red', alpha=0.3, label='Naive Interval')
    # Optionally, plot the bounds as lines:
    # ax.plot(x, naive_lower_bound, color='red', linewidth=1, linestyle='--')
    # ax.plot(x, naive_upper_bound, color='red', linewidth=1, linestyle='--')

    ax.set_title('Quantile Prediction Intervals')
    ax.set_xlabel('Sample Number')
    ax.set_ylabel(f'{TARGET_LABEL}')
    ax.grid(False)
    ax.set_facecolor('white')

    # Place the legend outside the plot
    ax.legend(loc='center left', bbox_to_anchor=(1.05, 0.5), borderaxespad=0.0)

    plt.tight_layout(rect=[0,0,0.80,1.0])  # Make space for legend

    # Save the plot to a BytesIO object
    img_data = BytesIO()
    img_data
    plt.savefig(img_data, format='png', bbox_inches='tight')
    plt.close()
    img_data.seek(0)

    # Add image to Excel
    if os.path.exists(file_path):
        wb = load_workbook(_ensure_excel_file(file_path))
        ws = wb.create_sheet(title="plot_quantile_intervals_to_excel")
    else:
        wb = Workbook()
        ws = wb.active
        ws.title = "plot_quantile_intervals_to_excel"

    img = Image(img_data)
    img.anchor = 'A1'
    ws.add_image(img)
    ws.column_dimensions['A'].width = 30
    _ensure_parent_dir(file_path)
    wb.save(_ensure_parent_dir(file_path))

# **Tuned Parameters**

In [18]:
# FIX K2: load the tuning results saved by hyperparameter_tuning/Optuna_autosampler.ipynb
# (was: a hand-pasted copy of an older tuning run that had been selected on the TEST set).
import pickle
if IN_KAGGLE:
    _bp = glob.glob("/kaggle/input/**/hyperparameter_tuning/best_params.pkl", recursive=True)[0]
else:
    _bp = "./drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/hyperparameter_tuning/best_params.pkl"
with open(_bp, "rb") as f:
    best_scores_autosampler = pickle.load(f)
print("Loaded tuning results from", _bp, "-", len(best_scores_autosampler), "model x pruner entries")


Loaded tuning results from /kaggle/input/datasets/daneshselwal/water-quality-wqi-tuned-params/hyperparameter_tuning/best_params.pkl - 80 model x pruner entries


In [19]:
best_scores_autosampler

{('Random Forest', 'MedianPruner'): {'best_score': 869.3671877788987,
  'cv_rmse': 29.48503328434443,
  'best_params': {'n_estimators': 200,
   'criterion': 'friedman_mse',
   'max_depth': None,
   'min_samples_split': 0.01,
   'min_samples_leaf': 3,
   'min_weight_fraction_leaf': 0.01,
   'max_features': 1.0,
   'max_leaf_nodes': None,
   'min_impurity_decrease': 0.2,
   'n_jobs': -1,
   'random_state': 42,
   'verbose': 0,
   'warm_start': False,
   'ccp_alpha': 0.1},
  'test_mse': 1071.6124042838674,
  'test_rmse': np.float64(32.735491508206614),
  'test_corr_coef': np.float64(0.9307870367802251),
  'pruner': 'MedianPruner'},
 ('Random Forest', 'NopPruner'): {'best_score': 869.3671877788987,
  'cv_rmse': 29.48503328434443,
  'best_params': {'n_estimators': 200,
   'criterion': 'friedman_mse',
   'max_depth': None,
   'min_samples_split': 0.01,
   'min_samples_leaf': 3,
   'min_weight_fraction_leaf': 0.01,
   'max_features': 1.0,
   'max_leaf_nodes': None,
   'min_impurity_decrease':

# **Quantile Regression with Lightgbm**

In [20]:
best_params = get_best_model_params(best_scores_autosampler, 'LightGBM')

# Define the quantiles you want to predict
quantiles = [0.05, 0.1, 0.15, 0.2, 0.3,
             0.4, 0.5, 0.6, 0.7, 0.8,
             0.85, 0.9, 0.95]

# Dictionary to store models for each quantile
models_quantile = {}

# DataFrame to store predictions
predictions_LGBM_df = pd.DataFrame()


# Train a model for each quantile and make predictions using LightGBM
for q in quantiles:
    # Create a copy of best_params to avoid modifying the original
    params = best_params.copy()

    # Update params with quantile-specific settings
    params.update({
        'objective': 'quantile',
        'alpha': q,
        'random_state': 42
    })

    # Initialize the model with the updated parameters
    model = LGBMRegressor(**params)

    # Fit the model
    model.fit(X_train, y_train)

    # Store the model
    models_quantile[q] = model

    # Make predictions
    predictions = model.predict(X_test)

    # Add predictions to the DataFrame
    predictions_LGBM_df[q] = predictions

# Add actual target values to the DataFrame
predictions_LGBM_df['Actual'] = y_test.values.ravel()

# Print the DataFrame with predictions for each quantile
print(predictions_LGBM_df.head())

         0.05         0.1        0.15         0.2         0.3         0.4  \
0  209.038569  282.407435  285.650518  297.974464  288.155648  292.290285   
1  117.205650  124.257850  118.932839  139.867696  149.334045  155.127084   
2  209.038569  281.870552  309.450227  328.754293  367.203845  379.409556   
3  139.277933  202.720184  195.639982  215.012368  243.310687  243.056459   
4  105.777171  118.875150  114.622574  125.671473  123.954822  125.474106   

          0.5         0.6         0.7         0.8        0.85         0.9  \
0  295.558597  289.538563  298.159602  295.542595  289.066816  299.871669   
1  172.332547  184.616929  209.396306  213.582091  195.982073  196.233358   
2  383.169687  413.868177  421.765944  450.315463  457.325661  446.356810   
3  237.108681  253.947297  228.488857  246.182874  279.819439  276.535851   
4  131.988415  139.571955  139.628553  134.920444  140.407050  135.751487   

         0.95  Actual  
0  317.592341     315  
1  206.890353     167  
2 

In [21]:
quantile_regression_print(predictions_LGBM_df,"./drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/quantile_regression/LightGBM.xlsx")

First few actual values: [315 167 516 144 127]
Coverage of 90% prediction interval: 85.37%


In [22]:
evaluation_features_print(predictions_LGBM_df,x_test,"./drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/quantile_regression/LightGBM.xlsx")

In [23]:
evaluate_uncertainity_matrix(predictions_LGBM_df, quantiles, "./drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/quantile_regression/LightGBM.xlsx", model_name='LightGBM')

Quantile crossing before sorting: 82 of 82 rows (100.0%)


{'empirical_quantiles': [np.float64(0.12195121951219512),
  np.float64(0.4024390243902439),
  np.float64(0.43902439024390244),
  np.float64(0.524390243902439),
  np.float64(0.6585365853658537),
  np.float64(0.7439024390243902),
  np.float64(0.8048780487804879),
  np.float64(0.8414634146341463),
  np.float64(0.8780487804878049),
  np.float64(0.8902439024390244),
  np.float64(0.9146341463414634),
  np.float64(0.9390243902439024),
  np.float64(0.975609756097561)],
 'average_interval_lengths': [np.float64(29.91772699950451),
  np.float64(54.32538806115461),
  np.float64(98.91502111003228)],
 'coverage_levels': [0.6, 0.8, 0.9],
 'average_nll': np.float64(4.751581459201829),
 'average_crps': np.float64(17.255672118914017),
 'nll_values': array([4.65220541, 4.31296752, 6.79143749, 7.36171485, 3.54616763,
        4.7827637 , 4.98572566, 4.95429886, 3.36877761, 4.29854472,
        4.30325349, 4.66197833, 3.48712322, 4.66929973, 3.70604707,
        4.77524704, 4.95923595, 3.78000089, 7.75689512,

In [24]:
calibrate_and_plot_intervals(predictions_LGBM_df, "./drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/quantile_regression/LightGBM.xlsx")

The empirical coverage before calibration is: 85.37%
The empirical coverage after calibration is: 85.37%


In [25]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

plot_quantile_intervals_to_excel(predictions_LGBM_df,"./drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/quantile_regression/LightGBM.xlsx")

# **Quantile Regression with XGBoost**

In [26]:
# Get the best hyperparameters for XGBRegressor
best_paramst = get_best_model_params(best_scores_autosampler, 'XGBoost')

# Define the quantiles you want to predict
quantiles = [0.05, 0.1, 0.15, 0.2, 0.3,
             0.4, 0.5, 0.6, 0.7, 0.8,
             0.85, 0.9, 0.95]

# Dictionary to store models for each quantile
models_quantile = {}

# DataFrame to store predictions
predictions_XGB_df = pd.DataFrame()

# Train a model for each quantile and make predictions
for q in quantiles:
    # Create a copy of best_params to avoid modifying the original
    params = best_paramst.copy()  # FIX Q1: was best_params (= LightGBM's settings from the cell above)

    # Adjust according to your XGBoost version
    params.update({
        'objective': 'reg:quantileerror',  # Use the correct objective
        'quantile_alpha': q,                         # Use 'alpha' to set the quantile level
        'random_state': 42
    })

    # Initialize the model with the updated parameters
    model = XGBRegressor(**params)

    # Fit the model
    model.fit(X_train, y_train)

    # Store the model
    models_quantile[q] = model

    # Make predictions
    predictions = model.predict(X_test)

    # Add predictions to the DataFrame
    predictions_XGB_df[q] = predictions

# Add actual target values to the DataFrame
predictions_XGB_df['Actual'] = y_test.values.ravel()

# Print the DataFrame with predictions for each quantile
print(predictions_XGB_df.head())

         0.05         0.1        0.15         0.2         0.3         0.4  \
0  124.038872  273.040894  288.856476  302.407196  316.199402  319.984344   
1  115.262924  125.283470  114.189453  132.616882  141.748322  153.963333   
2  124.038872  280.257416  312.270935  394.234863  399.328766  423.733215   
3  132.549332  208.535141  189.659821  201.766403  227.967590  239.792572   
4  115.711906  119.717705  117.256500  127.709717  131.676376  132.094833   

          0.5         0.6         0.7         0.8        0.85         0.9  \
0  287.593018  301.095367  300.653320  298.443359  330.329041  308.320587   
1  170.463211  178.454269  182.709518  198.260117  181.118576  188.762955   
2  425.091125  436.246246  426.910187  409.558746  416.305023  424.221924   
3  229.872437  251.594162  239.992874  244.228134  235.167862  238.120270   
4  135.206146  136.320023  132.278534  134.307037  138.021408  145.823151   

         0.95  Actual  
0  345.494873     315  
1  216.661667     167  
2 

In [27]:
quantile_regression_print(predictions_XGB_df,"./drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/quantile_regression/XGBoost.xlsx")

First few actual values: [315 167 516 144 127]
Coverage of 90% prediction interval: 87.80%


In [28]:
evaluation_features_print(predictions_XGB_df,x_test,"./drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/quantile_regression/XGBoost.xlsx")

In [29]:
evaluate_uncertainity_matrix(predictions_XGB_df, quantiles, "./drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/quantile_regression/XGBoost.xlsx", model_name='XGBoost')

Quantile crossing before sorting: 82 of 82 rows (100.0%)


{'empirical_quantiles': [np.float64(0.10975609756097561),
  np.float64(0.4146341463414634),
  np.float64(0.4878048780487805),
  np.float64(0.573170731707317),
  np.float64(0.6585365853658537),
  np.float64(0.7317073170731707),
  np.float64(0.8170731707317073),
  np.float64(0.8414634146341463),
  np.float64(0.8780487804878049),
  np.float64(0.926829268292683),
  np.float64(0.9512195121951219),
  np.float64(0.9634146341463414),
  np.float64(0.9878048780487805)],
 'average_interval_lengths': [np.float64(24.41340218520746),
  np.float64(51.6718125692228),
  np.float64(123.31082050974776)],
 'coverage_levels': [0.6, 0.8, 0.9],
 'average_nll': np.float64(4.770728652877145),
 'average_crps': np.float64(16.105380583739862),
 'nll_values': array([5.14969574, 4.36391222, 6.02356064, 7.23297144, 3.47233246,
        5.31876935, 4.82006021, 6.17730906, 3.83197059, 3.96029338,
        3.760532  , 4.35780874, 3.43338125, 4.62658517, 3.67237707,
        4.90821476, 4.24611812, 3.76101438, 7.13922346, 

In [30]:
calibrate_and_plot_intervals(predictions_XGB_df, "./drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/quantile_regression/XGBoost.xlsx")

The empirical coverage before calibration is: 85.37%
The empirical coverage after calibration is: 85.37%


In [31]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

plot_quantile_intervals_to_excel(predictions_XGB_df, "./drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/quantile_regression/XGBoost.xlsx")

# **Quantile Regression with GPBoost**

In [32]:
# Get the best hyperparameters for GPBoostRegressor
best_params = get_best_model_params(best_scores_autosampler, 'GPBoost')

# Define the quantiles you want to predict
quantiles = [0.05, 0.1, 0.15, 0.2, 0.3,
             0.4, 0.5, 0.6, 0.7, 0.8,
             0.85, 0.9, 0.95]

# Dictionary to store models for each quantile
models_quantile = {}

# DataFrame to store predictions
predictions_GPBoost_df = pd.DataFrame()

# Parameters that may conflict with quantile-specific settings
incompatible_keys = ['objective', 'alpha', 'quantile_alpha', 'boosting_type', 'metric', 'eval_metric']

# Train a model for each quantile and make predictions
for q in quantiles:
    # Create a copy of best_params to avoid modifying the original
    params = best_params.copy()

    # Remove incompatible parameters
    for key in incompatible_keys:
        params.pop(key, None)

    # Update params with quantile-specific settings
    params.update({
        'objective': 'quantile',
        'alpha': q,
        'random_state': 42
    })

    # Initialize the model with the updated parameters
    model = GPBoostRegressor(**params)

    # Fit the model
    model.fit(X_train, y_train)

    # Store the model
    models_quantile[q] = model

    # Make predictions
    predictions = model.predict(X_test)

    # Add predictions to the DataFrame
    predictions_GPBoost_df[q] = predictions

# Add actual target values to the DataFrame
predictions_GPBoost_df['Actual'] = y_test.values.ravel()

# Print the DataFrame with predictions for each quantile
print(predictions_GPBoost_df.head())

         0.05         0.1        0.15         0.2         0.3         0.4  \
0  250.889559  277.584736  314.628200  302.465713  300.766232  318.746179   
1  124.166261  121.384370  130.262570  153.266325  150.045603  176.413323   
2  250.889559  346.662260  380.077103  374.096456  394.854302  404.678127   
3  153.730401  195.430211  212.829058  225.350443  237.143950  234.541312   
4  119.342274  127.596399  128.929690  138.226025  123.025205  133.934038   

          0.5         0.6         0.7         0.8        0.85         0.9  \
0  302.828135  307.933582  299.698146  296.248652  303.675523  309.329731   
1  177.039982  169.105434  196.837526  213.366877  209.920319  214.043615   
2  409.325957  421.349192  455.685315  441.870414  450.262460  430.177418   
3  242.649755  227.812364  271.178758  222.700393  256.694178  241.778906   
4  135.581362  137.723431  136.203071  133.482425  146.922386  162.353212   

         0.95  Actual  
0  325.284681     315  
1  215.656636     167  
2 

In [33]:
quantile_regression_print(predictions_GPBoost_df,"./drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/quantile_regression/GPBoost.xlsx")

First few actual values: [315 167 516 144 127]
Coverage of 90% prediction interval: 82.93%


In [34]:
evaluation_features_print(predictions_GPBoost_df,x_test,"./drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/quantile_regression/GPBoost.xlsx")

In [35]:
evaluate_uncertainity_matrix(predictions_GPBoost_df, quantiles,"./drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/quantile_regression/GPBoost.xlsx", model_name='GPBoost')

Quantile crossing before sorting: 82 of 82 rows (100.0%)


{'empirical_quantiles': [np.float64(0.14634146341463414),
  np.float64(0.3902439024390244),
  np.float64(0.524390243902439),
  np.float64(0.5853658536585366),
  np.float64(0.6829268292682927),
  np.float64(0.7682926829268293),
  np.float64(0.8414634146341463),
  np.float64(0.8780487804878049),
  np.float64(0.8902439024390244),
  np.float64(0.926829268292683),
  np.float64(0.926829268292683),
  np.float64(0.9634146341463414),
  np.float64(0.975609756097561)],
 'average_interval_lengths': [np.float64(25.908374145307757),
  np.float64(54.7854902629789),
  np.float64(89.88200190421797)],
 'coverage_levels': [0.6, 0.8, 0.9],
 'average_nll': np.float64(5.000032655397844),
 'average_crps': np.float64(17.31823609828266),
 'nll_values': array([ 4.18237743,  4.32827837,  6.51826825,  7.70990278,  3.70499113,
         4.71838147,  5.35285005,  4.00231029,  3.74776   ,  4.80627503,
         3.843188  ,  5.33124554,  3.22036906,  4.54284945,  4.18328428,
         4.62906423,  5.84215236,  3.7186208

In [36]:
calibrate_and_plot_intervals(predictions_GPBoost_df,"./drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/quantile_regression/GPBoost.xlsx")

The empirical coverage before calibration is: 85.37%
The empirical coverage after calibration is: 90.24%


In [37]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

plot_quantile_intervals_to_excel(predictions_GPBoost_df,"./drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/quantile_regression/GPBoost.xlsx")

# **Quantile Regression with NGBoost**

In [38]:
# Correct best_params assignment
best_params = get_best_model_params(best_scores_autosampler, 'NGBoost')

# DataFrame to store predictions
predictions_NGB_df = pd.DataFrame()

# Remove 'Score' and 'Dist' if present in best_params
for key in ['Score', 'Dist']:
    best_params.pop(key, None)

# Update params with model settings
best_params.update({
    'Dist': Normal,
    'Score': LogScore,
    'random_state': 42,
    'verbose': True
})

# Initialize and fit the NGBoost model once
model = NGBRegressor(**best_params)

# Fit the model
model.fit(X_train, y_train)

# Make predictions
pred_dist = model.pred_dist(X_test)

# Extract the mean (mu) and standard deviation (sigma) of the predicted distributions
mu = pred_dist.loc    # Mean predictions
sigma = pred_dist.scale  # Standard deviation predictions

# Define the quantiles you want to predict
quantiles = [0.05, 0.1, 0.15, 0.2, 0.3,
             0.4, 0.5, 0.6, 0.7, 0.8,
             0.85, 0.9, 0.95]

# For each quantile, compute predictions
for q in quantiles:
    # Calculate the z-score for the desired quantile
    z = norm.ppf(q)

    # Compute the quantile prediction per sample
    predictions = mu + z * sigma

    # Add predictions to the DataFrame
    predictions_NGB_df[q] = predictions

# Add actual target values to the DataFrame
predictions_NGB_df['Actual'] = y_test.values.ravel()

# Print the DataFrame with predictions for each quantile
print(predictions_NGB_df.head())

[iter 0] loss=5.8251 val_loss=0.0000 scale=1.0000 norm=65.2323


[iter 100] loss=2.9871 val_loss=0.0000 scale=2.0000 norm=5.0425


[iter 200] loss=1.3146 val_loss=0.0000 scale=1.0000 norm=0.9473


[iter 300] loss=0.6121 val_loss=0.0000 scale=1.0000 norm=0.6022


[iter 400] loss=0.1019 val_loss=0.0000 scale=1.0000 norm=0.4375


         0.05         0.1        0.15         0.2         0.3         0.4  \
0  295.832089  295.944617  296.020539  296.080879  296.179134  296.263088   
1  209.287271  209.371802  209.428834  209.474162  209.547971  209.611038   
2  435.240432  435.309729  435.356482  435.393641  435.454148  435.505848   
3  217.493201  217.579864  217.638336  217.684807  217.760479  217.825137   
4  134.475594  134.524312  134.557183  134.583307  134.625846  134.662195   

          0.5         0.6         0.7         0.8        0.85         0.9  \
0  296.341559  296.420030  296.503985  296.602239  296.662580  296.738501   
1  209.669986  209.728933  209.792000  209.865809  209.911137  209.968169   
2  435.554172  435.602495  435.654196  435.714702  435.751861  435.798615   
3  217.885572  217.946006  218.010665  218.086336  218.132807  218.191279   
4  134.696168  134.730142  134.766491  134.809030  134.835154  134.868024   

         0.95  Actual  
0  296.851029     315  
1  210.052700     167  
2 

In [39]:
quantile_regression_print(predictions_NGB_df,"./drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/quantile_regression/NGBoost.xlsx")

First few actual values: [315 167 516 144 127]
Coverage of 90% prediction interval: 2.44%


In [40]:
evaluation_features_print(predictions_NGB_df,x_test,"./drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/quantile_regression/NGBoost.xlsx")

In [41]:
evaluate_uncertainity_matrix(predictions_NGB_df, quantiles,"./drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/quantile_regression/NGBoost.xlsx", model_name='NGBoost')

Quantile crossing before sorting: 0 of 82 rows (0.0%)


{'empirical_quantiles': [np.float64(0.8414634146341463),
  np.float64(0.8414634146341463),
  np.float64(0.8536585365853658),
  np.float64(0.8536585365853658),
  np.float64(0.8658536585365854),
  np.float64(0.8658536585365854),
  np.float64(0.8658536585365854),
  np.float64(0.8658536585365854),
  np.float64(0.8658536585365854),
  np.float64(0.8658536585365854),
  np.float64(0.8658536585365854),
  np.float64(0.8658536585365854),
  np.float64(0.8658536585365854)],
 'average_interval_lengths': [np.float64(0.36879516461842093),
  np.float64(0.561570932063547),
  np.float64(0.7207684881588903)],
 'coverage_levels': [0.6, 0.8, 0.9],
 'average_nll': np.float64(10742.891218551242),
 'average_crps': np.float64(22.47742642568558),
 'nll_values': array([1.81416754e+03, 1.68153325e+04, 8.89385758e+04, 4.79673074e+04,
        1.64578846e+03, 2.63364771e+04, 9.08648755e+03, 9.11025952e+02,
        4.65955551e+03, 7.81548303e+03, 9.66978848e+02, 3.66528451e+04,
        1.33955648e+02, 3.54890328e+03, 

In [42]:
calibrate_and_plot_intervals(predictions_NGB_df,"./drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/quantile_regression/NGBoost.xlsx")

The empirical coverage before calibration is: 4.88%
The empirical coverage after calibration is: 100.00%


In [43]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

plot_quantile_intervals_to_excel(predictions_NGB_df,"./drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/quantile_regression/NGBoost.xlsx")

# **Quantile Regression with Gradient Boosting**

In [44]:
best_params = get_best_model_params(best_scores_autosampler, 'GradientBoosting')


# Define the quantiles you want to predict
quantiles = [0.05, 0.1, 0.15, 0.2, 0.3,
             0.4, 0.5, 0.6, 0.7, 0.8,
             0.85, 0.9, 0.95]

# Dictionary to store models for each quantile
models_quantile = {}

# DataFrame to store predictions
predictions_GBR_df = pd.DataFrame()

# Custom quantile loss function
def quantile_loss(q, y_true, y_pred):
    e = y_true - y_pred
    return np.mean(np.maximum(q * e, (q - 1) * e))

# Train a model for each quantile and make predictions using GradientBoostingRegressor
for q in quantiles:

    params = best_params.copy()

    params.update({
        'loss': 'quantile',
        'alpha': q,
        'random_state': 42
    })
    # Initialize the model with quantile-specific settings
    model = GradientBoostingRegressor(**params)

    # Fit the model
    model.fit(X_train, y_train)

    # Store the model
    models_quantile[q] = model

    # Make predictions
    predictions = model.predict(X_test)

    # Add predictions to the DataFrame
    predictions_GBR_df[q] = predictions

# Add actual target values to the DataFrame
predictions_GBR_df['Actual'] = y_test.values.ravel()

# Print the DataFrame with predictions for each quantile
print(predictions_GBR_df.head())

         0.05         0.1        0.15         0.2         0.3         0.4  \
0  132.738788  185.621236  215.180423  227.949890  298.546001  302.292641   
1  118.015576  126.060137  130.136623  127.109379  131.415161  158.631698   
2  132.738788  218.149489  212.715868  227.028532  319.200543  371.045301   
3  132.516024  181.065014  191.211750  191.007837  234.198932  239.263041   
4  124.359458  126.767564  136.905149  124.253141  132.402482  133.249698   

          0.5         0.6         0.7         0.8        0.85         0.9  \
0  300.266562  300.812605  301.173253  295.610963  299.147580  298.107437   
1  167.191615  183.877709  199.292272  232.060970  237.480040  211.634356   
2  368.861257  401.809951  432.841535  444.853909  462.813128  420.047086   
3  234.545531  258.826423  228.098695  211.010678  252.616253  248.032549   
4  133.029978  139.860271  136.449271  152.783657  151.417008  144.995638   

         0.95  Actual  
0  317.675931     315  
1  260.298101     167  
2 

In [45]:
quantile_regression_print(predictions_GBR_df,"./drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/quantile_regression/Gradient Boosting.xlsx")

First few actual values: [315 167 516 144 127]
Coverage of 90% prediction interval: 79.27%


In [46]:
evaluation_features_print(predictions_GBR_df,x_test,"./drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/quantile_regression/Gradient Boosting.xlsx")

In [47]:
evaluate_uncertainity_matrix(predictions_GBR_df, quantiles,"./drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/quantile_regression/Gradient Boosting.xlsx", model_name='GBM')

Quantile crossing before sorting: 82 of 82 rows (100.0%)


{'empirical_quantiles': [np.float64(0.18292682926829268),
  np.float64(0.36585365853658536),
  np.float64(0.4634146341463415),
  np.float64(0.5121951219512195),
  np.float64(0.5853658536585366),
  np.float64(0.7317073170731707),
  np.float64(0.7804878048780488),
  np.float64(0.8170731707317073),
  np.float64(0.8658536585365854),
  np.float64(0.9024390243902439),
  np.float64(0.9146341463414634),
  np.float64(0.9512195121951219),
  np.float64(0.975609756097561)],
 'average_interval_lengths': [np.float64(42.33898473003078),
  np.float64(72.38868583241535),
  np.float64(133.37479734507957)],
 'coverage_levels': [0.6, 0.8, 0.9],
 'average_nll': np.float64(4.851500624847231),
 'average_crps': np.float64(17.652305284183974),
 'nll_values': array([4.99098902, 4.68596446, 6.57103745, 7.32624692, 4.32036939,
        5.22688897, 4.9380679 , 4.1859242 , 4.5223273 , 4.4549706 ,
        4.21804172, 4.59830001, 4.39320888, 4.69456036, 3.89125959,
        4.73766316, 4.60725311, 3.82686865, 7.4709136

In [48]:
calibrate_and_plot_intervals(predictions_GBR_df,"./drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/quantile_regression/Gradient Boosting.xlsx")

The empirical coverage before calibration is: 78.05%
The empirical coverage after calibration is: 92.68%


In [49]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

plot_quantile_intervals_to_excel(predictions_GBR_df,"./drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/quantile_regression/Gradient Boosting.xlsx")

# **Quantile Regression with CatBoost**

In [50]:
# Get the best hyperparameters for CatBoostRegressor
best_params = get_best_model_params(best_scores_autosampler, 'CatBoost')

# Define the quantiles you want to predict
quantiles = [0.05, 0.1, 0.15, 0.2, 0.3,
             0.4, 0.5, 0.6, 0.7, 0.8,
             0.85, 0.9, 0.95]

# Dictionary to store models for each quantile
models_quantile = {}

# DataFrame to store predictions
predictions_CatBoost_df = pd.DataFrame()

# Parameters that may conflict with quantile-specific settings
incompatible_keys = ['objective', 'loss_function', 'eval_metric']

# Train a model for each quantile and make predictions
for q in quantiles:
    # Create a copy of best_params to avoid modifying the original
    params = best_params.copy()

    # Remove incompatible parameters
    for key in incompatible_keys:
        params.pop(key, None)

    # Update params with quantile-specific settings
    params.update({
        'loss_function': 'Quantile:alpha={}'.format(q),
        'random_seed': 42
    })

    # Initialize the model with the updated parameters
    model = CatBoostRegressor(**params)

    # Fit the model
    model.fit(X_train, y_train, verbose=0)

    # Store the model
    models_quantile[q] = model

    # Make predictions
    predictions = model.predict(X_test)

    # Add predictions to the DataFrame
    predictions_CatBoost_df[q] = predictions

# Add actual target values to the DataFrame
predictions_CatBoost_df['Actual'] = y_test.values.ravel()

# Print the DataFrame with predictions for each quantile
print(predictions_CatBoost_df.head())

         0.05         0.1        0.15         0.2         0.3         0.4  \
0  281.934004  285.929023  297.196177  298.856744  287.073883  292.689302   
1  150.073507  123.784128  141.474423  143.976991  162.560547  179.590324   
2  420.103051  354.672102  349.301148  408.526872  435.818791  414.781206   
3  211.810405  239.805609  234.491351  249.535661  245.198831  230.579503   
4  140.833617  127.503609  127.344048  145.361079  130.133643  131.855752   

          0.5         0.6         0.7         0.8        0.85         0.9  \
0  287.580909  296.411627  285.382620  288.079779  300.838766  303.887455   
1  166.770908  166.845779  165.393527  168.333489  166.299530  155.000042   
2  424.700853  426.462537  430.327661  420.355140  437.653317  445.520575   
3  209.062740  209.607070  236.977080  235.602510  240.910528  253.552214   
4  131.829752  128.979517  121.491448  132.878883  126.627621  113.560558   

         0.95  Actual  
0  292.269616     315  
1  140.209293     167  
2 

In [51]:
quantile_regression_print(predictions_CatBoost_df,"./drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/quantile_regression/CatBoost.xlsx")

First few actual values: [315 167 516 144 127]
Coverage of 90% prediction interval: 51.22%


In [52]:
evaluation_features_print(predictions_CatBoost_df,x_test,"./drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/quantile_regression/CatBoost.xlsx")

In [53]:
evaluate_uncertainity_matrix(predictions_CatBoost_df, quantiles,"./drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/quantile_regression/CatBoost.xlsx", model_name='CatBoost')

Quantile crossing before sorting: 82 of 82 rows (100.0%)


{'empirical_quantiles': [np.float64(0.4146341463414634),
  np.float64(0.4878048780487805),
  np.float64(0.573170731707317),
  np.float64(0.6341463414634146),
  np.float64(0.6707317073170732),
  np.float64(0.7073170731707317),
  np.float64(0.7317073170731707),
  np.float64(0.7439024390243902),
  np.float64(0.7804878048780488),
  np.float64(0.7926829268292683),
  np.float64(0.8536585365853658),
  np.float64(0.9146341463414634),
  np.float64(0.926829268292683)],
 'average_interval_lengths': [np.float64(16.99140198308553),
  np.float64(36.563130888142055),
  np.float64(53.63376668425492)],
 'coverage_levels': [0.6, 0.8, 0.9],
 'average_nll': np.float64(5.692054281218272),
 'average_crps': np.float64(17.131699241968565),
 'nll_values': array([ 8.61792541,  3.78426838,  9.16660449, 26.46293324,  3.24016565,
         3.67439479,  4.89995854,  6.96708721,  3.17321608,  3.78126436,
         3.02684113,  4.96315439,  2.57862138,  4.51374931,  4.17463413,
         4.54201354,  5.28743672,  3.5385

In [54]:
calibrate_and_plot_intervals(predictions_CatBoost_df,"./drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/quantile_regression/CatBoost.xlsx")

The empirical coverage before calibration is: 56.10%
The empirical coverage after calibration is: 100.00%


In [55]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

plot_quantile_intervals_to_excel(predictions_CatBoost_df,"./drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/quantile_regression/CatBoost.xlsx")

# **Quantile Regression with HistGradientBoosting**

In [56]:

# Assume best_params is a dictionary containing the best parameters for HistGradientBoostingRegressor
best_params = get_best_model_params(best_scores_autosampler, 'HGBR')

# Define the quantiles you want to predict
quantiles = [0.05, 0.1, 0.15, 0.2, 0.3,
             0.4, 0.5, 0.6, 0.7, 0.8,
             0.85, 0.9, 0.95]

# Dictionary to store models for each quantile
models_quantile = {}

# DataFrame to store predictions
predictions_HGBR_df = pd.DataFrame()

# Train a model for each quantile and make predictions using HistGradientBoostingRegressor
for q in quantiles:
    # Create a copy of best_params to avoid modifying the original
    params = best_params.copy()

    # Update params with quantile-specific settings
    params.update({
        'loss': 'quantile',
        'quantile': q,
        'random_state': 42
    })

    # Initialize the model with the updated parameters
    model = HistGradientBoostingRegressor(**params)

    # Fit the model
    model.fit(X_train, y_train)

    # Store the model
    models_quantile[q] = model

    # Make predictions
    predictions = model.predict(X_test)

    # Add predictions to the DataFrame
    predictions_HGBR_df[q] = predictions

# Add actual target values to the DataFrame
predictions_HGBR_df['Actual'] = y_test.values.ravel()

# Print the DataFrame with predictions for each quantile
print(predictions_HGBR_df.head())

         0.05         0.1        0.15         0.2         0.3         0.4  \
0  259.605969  288.027821  283.802766  294.298093  299.008769  292.636713   
1  109.296113  119.544815  122.254413  126.989688  128.734416  156.273732   
2  259.605969  301.656390  293.211355  333.977123  334.595991  350.524610   
3  159.162059  191.052293  202.146436  212.669217  229.737630  226.563979   
4  121.799474  118.069953  118.843624  126.497832  123.629628  132.835095   

          0.5         0.6         0.7         0.8        0.85         0.9  \
0  298.811470  303.536401  310.715740  296.700880  322.806359  319.972259   
1  178.072157  195.775383  199.541273  240.744448  253.656973  247.206255   
2  352.422256  361.618210  375.894448  395.211593  419.534686  438.099840   
3  227.859728  238.508678  220.377338  219.074876  282.696075  268.516595   
4  140.991765  143.912678  141.164468  154.047961  151.863985  154.287015   

         0.95  Actual  
0  332.514134     315  
1  244.324643     167  
2 

In [57]:
quantile_regression_print(predictions_HGBR_df,"./drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/quantile_regression/HGBM.xlsx")

First few actual values: [315 167 516 144 127]
Coverage of 90% prediction interval: 62.20%


In [58]:
evaluation_features_print(predictions_HGBR_df,x_test,"./drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/quantile_regression/HGBM.xlsx")

In [59]:
evaluate_uncertainity_matrix(predictions_HGBR_df, quantiles,"./drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/quantile_regression/HGBM.xlsx", model_name='HGBM')

Quantile crossing before sorting: 82 of 82 rows (100.0%)


{'empirical_quantiles': [np.float64(0.34146341463414637),
  np.float64(0.4634146341463415),
  np.float64(0.5),
  np.float64(0.6219512195121951),
  np.float64(0.6829268292682927),
  np.float64(0.7804878048780488),
  np.float64(0.8170731707317073),
  np.float64(0.8414634146341463),
  np.float64(0.8780487804878049),
  np.float64(0.9024390243902439),
  np.float64(0.9390243902439024),
  np.float64(0.9512195121951219),
  np.float64(0.9634146341463414)],
 'average_interval_lengths': [np.float64(30.923936025412843),
  np.float64(55.358817713455544),
  np.float64(74.32890691990676)],
 'coverage_levels': [0.6, 0.8, 0.9],
 'average_nll': np.float64(5.112048709392283),
 'average_crps': np.float64(18.80664570510608),
 'nll_values': array([ 4.28411673,  4.73228707,  9.45719644,  6.96174539,  4.05768988,
         4.82919863,  5.35797626,  4.2380664 ,  3.91310164,  3.93663085,
         4.05288018,  5.3876061 ,  3.83629877,  5.05057405,  4.18938586,
         4.62922386,  5.90714296,  3.97257754,  9.510

In [60]:
calibrate_and_plot_intervals(predictions_HGBR_df, "./drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/quantile_regression/HGBM.xlsx")

The empirical coverage before calibration is: 65.85%
The empirical coverage after calibration is: 92.68%


In [61]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

plot_quantile_intervals_to_excel(predictions_HGBR_df, "./drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/quantile_regression/HGBM.xlsx")

# **Quantile Regression with TabNet**

In [62]:
best_params = get_best_model_params(best_scores_autosampler, 'TabNet')

# Quantiles
quantiles = [0.05, 0.1, 0.15, 0.2, 0.3,
             0.4, 0.5, 0.6, 0.7, 0.8,
             0.85, 0.9, 0.95]

# Storage
models_quantile = {}
predictions_TabNet_df = pd.DataFrame()

# Convert data to numpy (TabNet requirement)
X_train_np = np.array(X_train)
y_train_np = np.array(y_train).reshape(-1, 1)
X_test_np  = np.array(X_test)

# Quantile loss (pinball loss)
def quantile_loss(q):
    def loss(y_pred, y_true):
        error = y_true - y_pred
        return torch.mean(torch.maximum(q * error, (q - 1) * error))
    return loss

# FIX Q3: early stopping on a 20% validation split of TRAIN, scored with the pinball loss of
# the quantile being trained (was: eval_set = the training data itself, scored with MSE,
# so early stopping never triggered).
from pytorch_tabnet.metrics import Metric
X_fit_np, X_val_np, y_fit_np, y_val_np = train_test_split(X_train_np, y_train_np, test_size=0.2, random_state=42)

def pinball_metric(q):
    class Pinball(Metric):
        def __init__(self):
            self._name = f'pinball_{q}'
            self._maximize = False
        def __call__(self, y_true, y_score, weights=None):  # FIX B8: pytorch-tabnet2 also passes sample weights
            e = np.asarray(y_true, dtype=float) - np.asarray(y_score, dtype=float)
            return float(np.mean(np.maximum(q * e, (q - 1) * e)))
    return Pinball

# Train one TabNet per quantile
for q in quantiles:
    model = TabNetRegressor(
        **best_params,
    )

    model.fit(
        X_fit_np,
        y_fit_np,
        eval_set=[(X_val_np, y_val_np)],
        eval_metric=[pinball_metric(q)],
        loss_fn=quantile_loss(q),
        max_epochs=200,
        patience=30,
        batch_size=1024,
        virtual_batch_size=128,
        num_workers=0,
        drop_last=False
    )

    # Store model
    models_quantile[q] = model

    # Predict
    preds = model.predict(X_test_np).ravel()
    predictions_TabNet_df[q] = preds

# Add actual values
predictions_TabNet_df['Actual'] = y_test.values.ravel()

print(predictions_TabNet_df.head())


Early stopping occurred at epoch 66 with best_epoch = 36 and best_val_0_pinball_0.05 = 2.97609



Early stopping occurred at epoch 88 with best_epoch = 58 and best_val_0_pinball_0.1 = 5.97858



Early stopping occurred at epoch 110 with best_epoch = 80 and best_val_0_pinball_0.15 = 7.58731



Early stopping occurred at epoch 64 with best_epoch = 34 and best_val_0_pinball_0.2 = 10.24476



Early stopping occurred at epoch 123 with best_epoch = 93 and best_val_0_pinball_0.3 = 8.96689



Early stopping occurred at epoch 178 with best_epoch = 148 and best_val_0_pinball_0.4 = 5.09008


... (3 outputs left out)


Early stopping occurred at epoch 150 with best_epoch = 120 and best_val_0_pinball_0.8 = 3.2703



Early stopping occurred at epoch 99 with best_epoch = 69 and best_val_0_pinball_0.85 = 5.68619



Early stopping occurred at epoch 131 with best_epoch = 101 and best_val_0_pinball_0.9 = 4.53733



Early stopping occurred at epoch 143 with best_epoch = 113 and best_val_0_pinball_0.95 = 2.80836
         0.05         0.1        0.15         0.2         0.3         0.4  \
0  201.498505  273.570526  294.356140  129.809723  250.892456  305.666351   
1   81.836853  108.960617  113.879738   74.653503  133.900162  132.470917   
2  295.898926  420.355896  729.798157  273.450317  647.122314  664.453125   
3  108.080879  216.862823  163.790588  159.489471  190.118286  196.836060   
4   61.312149  110.399734  103.149078  121.480591  101.954071  107.937714   

          0.5         0.6         0.7         0.8        0.85         0.9  \
0  347.287415  303.075134  280.948059  295.816528  308.144043  320.390228   
1  136.709351  113.468948  147.585648  132.227386  126.284286  125.445282   
2  604.765991  561.358215  611.847778  537.189209  550.215454  717.257324   
3  212.040009  228.402603  216.620728  237.721146  337.558838  357.310730   
4  113.960739  125.925903   93.531006  132.876373  105

In [63]:
quantile_regression_print(predictions_TabNet_df,"./drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/quantile_regression/TabNet.xlsx")

First few actual values: [315 167 516 144 127]
Coverage of 90% prediction interval: 92.68%


In [64]:
evaluation_features_print(predictions_TabNet_df,x_test,"./drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/quantile_regression/TabNet.xlsx")

In [65]:
evaluate_uncertainity_matrix(predictions_TabNet_df, quantiles, "./drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/quantile_regression/TabNet.xlsx", model_name='TabNet')

Quantile crossing before sorting: 82 of 82 rows (100.0%)


{'empirical_quantiles': [np.float64(0.024390243902439025),
  np.float64(0.07317073170731707),
  np.float64(0.1951219512195122),
  np.float64(0.2804878048780488),
  np.float64(0.3902439024390244),
  np.float64(0.4878048780487805),
  np.float64(0.5609756097560976),
  np.float64(0.6463414634146342),
  np.float64(0.6951219512195121),
  np.float64(0.7560975609756098),
  np.float64(0.8536585365853658),
  np.float64(0.8902439024390244),
  np.float64(0.9512195121951219)],
 'average_interval_lengths': [np.float64(29.317817548426188),
  np.float64(69.6808114400724),
  np.float64(104.94616140970369)],
 'coverage_levels': [0.6, 0.8, 0.9],
 'average_nll': np.float64(4.604160504843505),
 'average_crps': np.float64(13.496088057494747),
 'nll_values': array([5.15233889, 5.49944474, 6.10691028, 5.64980149, 4.38265685,
        5.23279272, 4.08049161, 4.75379105, 3.86403281, 4.05835832,
        4.05420057, 4.40003227, 4.31819484, 4.2206289 , 4.13113791,
        4.16958837, 3.87615379, 4.46941237, 5.99527

In [66]:
calibrate_and_plot_intervals(predictions_TabNet_df, "./drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/quantile_regression/TabNet.xlsx")

The empirical coverage before calibration is: 92.68%
The empirical coverage after calibration is: 92.68%


In [67]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

plot_quantile_intervals_to_excel(predictions_TabNet_df,"./drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/quantile_regression/TabNet.xlsx")

# **Quantile Regression with PGBM**

In [68]:
class PGBMWrapper(BaseEstimator, RegressorMixin):
    def __init__(self, **params):
        self.params = params
        self.model = None

    def fit(self, X, y):
        X_ = X.to_numpy() if hasattr(X, "to_numpy") else np.array(X)
        y_ = y.to_numpy() if hasattr(y, "to_numpy") else np.array(y)
        self.model = PGBM()
        self.model.train(
            train_set=(X_, y_),
            objective=mseloss_objective,
            metric=rmseloss_metric,
            params=self.params
        )
        return self

    def predict(self, X):
        X_ = X.to_numpy() if hasattr(X, "to_numpy") else np.array(X)
        return self.model.predict(X_).numpy()


In [69]:
# Assuming best_params is obtained correctly
best_params = get_best_model_params(best_scores_autosampler, 'PGBM')

# Remove any conflicting parameters from best_params
incompatible_keys = ['Dist', 'Score']
for key in incompatible_keys:
    best_params.pop(key, None)

# Initialize PGBM model
pgbm_model = PGBM()
def mseloss_objective(yhat, y, sample_weight=None):
    # Ensure that yhat and y are PyTorch tensors
    if not torch.is_tensor(yhat):
        yhat = torch.from_numpy(np.array(yhat)).float()
    if not torch.is_tensor(y):
        y = torch.from_numpy(np.array(y)).float()
    gradient = yhat - y
    hessian = torch.ones_like(yhat)
    return gradient, hessian

def rmseloss_metric(yhat, y, sample_weight=None):
    # Ensure that yhat and y are PyTorch tensors
    if not torch.is_tensor(yhat):
        yhat = torch.from_numpy(np.array(yhat)).float()
    if not torch.is_tensor(y):
        y = torch.from_numpy(np.array(y)).float()
    loss = torch.sqrt(torch.mean((yhat - y) ** 2))
    return loss

X_train = np.array(X_train)
y_train = np.array(y_train)
X_test = np.array(X_test)
y_test = np.array(y_test)

# Fit the model
pgbm_model.train((X_train, y_train), objective=mseloss_objective, metric=rmseloss_metric, params=best_params)

# Predict the distribution
pred_dist = pgbm_model.predict_dist(X_test)

# Define the quantiles you want to predict
quantiles = [0.05, 0.1, 0.15, 0.2, 0.3,
             0.4, 0.5, 0.6, 0.7, 0.8,
             0.85, 0.9, 0.95]

# DataFrame to store predictions
predictions_PGBM_df = pd.DataFrame()

# Calculate and store quantiles
for q in quantiles:
    predictions_PGBM_df[q] = np.quantile(pred_dist, q, axis=0)

# Add actual target values to the DataFrame
predictions_PGBM_df['Actual'] = y_test.ravel()

# Print the DataFrame with predictions for each quantile
print(predictions_PGBM_df.head())

Training on CPU


Estimator 0/300, Train metric: 78.9001
Estimator 1/300, Train metric: 76.0510
Estimator 2/300, Train metric: 73.3512
Estimator 3/300, Train metric: 70.6507
Estimator 4/300, Train metric: 68.2775


Estimator 5/300, Train metric: 65.8602
Estimator 6/300, Train metric: 63.4633
Estimator 7/300, Train metric: 61.1513
Estimator 8/300, Train metric: 59.1168
Estimator 9/300, Train metric: 57.0578
Estimator 10/300, Train metric: 55.1212
Estimator 11/300, Train metric: 53.2731


Estimator 12/300, Train metric: 51.4449
Estimator 13/300, Train metric: 49.6976
Estimator 14/300, Train metric: 48.0189
Estimator 15/300, Train metric: 46.4079
Estimator 16/300, Train metric: 44.8642
Estimator 17/300, Train metric: 43.4459
Estimator 18/300, Train metric: 42.0784


Estimator 19/300, Train metric: 40.7679
Estimator 20/300, Train metric: 39.4945
Estimator 21/300, Train metric: 38.2714
Estimator 22/300, Train metric: 37.0826
Estimator 23/300, Train metric: 35.9078
Estimator 24/300, Train metric: 34.7604


Estimator 25/300, Train metric: 33.7132
Estimator 26/300, Train metric: 32.6903
Estimator 27/300, Train metric: 31.6972
Estimator 28/300, Train metric: 30.7656
Estimator 29/300, Train metric: 29.8942
Estimator 30/300, Train metric: 29.0440


... (59 outputs left out)

Estimator 286/300, Train metric: 1.5873
Estimator 287/300, Train metric: 1.5789
Estimator 288/300, Train metric: 1.5702
Estimator 289/300, Train metric: 1.5613
Estimator 290/300, Train metric: 1.5522
Estimator 291/300, Train metric: 1.5438


Estimator 292/300, Train metric: 1.5344
Estimator 293/300, Train metric: 1.5256
Estimator 294/300, Train metric: 1.5180
Estimator 295/300, Train metric: 1.5104
Estimator 296/300, Train metric: 1.5022
Estimator 297/300, Train metric: 1.4947


Estimator 298/300, Train metric: 1.4865
Estimator 299/300, Train metric: 1.4789


         0.05         0.1        0.15         0.2         0.3         0.4  \
0  273.770142  279.715088  283.416382  286.174072  288.616516  291.302368   
1  176.622406  177.881943  179.509827  179.796646  180.421417  180.856171   
2  404.486481  411.515717  417.013214  419.941040  424.265442  427.468536   
3  215.405060  216.753906  217.368576  217.773972  219.425995  220.388596   
4  128.763931  130.137955  131.065704  131.442856  132.016235  132.567017   

          0.5         0.6         0.7         0.8        0.85         0.9  \
0  291.961761  293.057007  294.430511  297.883331  300.085022  302.833832   
1  181.357117  181.681503  182.287033  183.428619  183.920181  184.590027   
2  429.695435  432.362762  436.334686  440.495880  441.710754  446.060730   
3  220.905426  221.782654  222.771255  224.122360  224.731979  225.387131   
4  133.197998  133.691971  134.486389  135.446228  135.856003  136.302917   

         0.95  Actual  
0  312.078064     315  
1  184.846405     167  
2 

In [70]:
quantile_regression_print(predictions_PGBM_df,"./drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/quantile_regression/PGBM.xlsx")

First few actual values: [315 167 516 144 127]
Coverage of 90% prediction interval: 15.85%


In [71]:
evaluation_features_print(predictions_PGBM_df,x_test,"./drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/quantile_regression/PGBM.xlsx")

In [72]:
evaluate_uncertainity_matrix(predictions_PGBM_df, quantiles, "./drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/quantile_regression/PGBM.xlsx", model_name='PGBM')

Quantile crossing before sorting: 0 of 82 rows (0.0%)


{'empirical_quantiles': [np.float64(0.7317073170731707),
  np.float64(0.7804878048780488),
  np.float64(0.7804878048780488),
  np.float64(0.7926829268292683),
  np.float64(0.8048780487804879),
  np.float64(0.8048780487804879),
  np.float64(0.8170731707317073),
  np.float64(0.8292682926829268),
  np.float64(0.8292682926829268),
  np.float64(0.8292682926829268),
  np.float64(0.8414634146341463),
  np.float64(0.8536585365853658),
  np.float64(0.8902439024390244)],
 'average_interval_lengths': [np.float64(5.55725069743831),
  np.float64(9.496273226854278),
  np.float64(13.634565679038444)],
 'coverage_levels': [0.6, 0.8, 0.9],
 'average_nll': np.float64(38.95542851179763),
 'average_crps': np.float64(20.910419995145098),
 'nll_values': array([  5.33086439,  18.32641976,  21.65288968, 182.24992579,
          4.9276672 ,   4.76181008,  75.10835875,  14.50496157,
          4.76299257,  36.80422509,   6.14645296,   8.91971305,
          3.40556539,  43.04965821,   3.00310625,  28.85294257,
   

In [73]:
calibrate_and_plot_intervals(predictions_PGBM_df, "./drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/quantile_regression/PGBM.xlsx")

The empirical coverage before calibration is: 17.07%
The empirical coverage after calibration is: 92.68%


In [74]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

plot_quantile_intervals_to_excel(predictions_PGBM_df,"./drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/quantile_regression/PGBM.xlsx")